In [1]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class Document:
    content: str
    file_path: str
    file_type: str
    metadata: dict

In [2]:
document = Document(
    content="Hello RAG",
    file_path="example.txt",
    file_type=".txt",
    metadata={}
)

print(document)

Document(content='Hello RAG', file_path='example.txt', file_type='.txt', metadata={})


In [3]:
from pypdf import PdfReader
from docx import Document as DocxDocument


def load_pdf(file_path):
    reader = PdfReader(file_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        if text.strip():
            pages.append({
                "page_number": page_number,
                "content": text
            })

    return pages


def load_docx(file_path):
    doc = DocxDocument(file_path)

    paragraphs = []

    for paragraph in doc.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    return "\n".join(paragraphs)

In [4]:
def load_text(file_path):
    path = Path(file_path)

    return path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

In [5]:
def load_document(file_path):

    path = Path(file_path)

    extension = path.suffix.lower()

    if extension == ".pdf":

        pages = load_pdf(path)

        return Document(
            content="\n\n".join(
                page["content"]
                for page in pages
            ),
            file_path=str(path),
            file_type=extension,
            metadata={
                "pages": pages
            }
        )

    elif extension == ".docx":

        content = load_docx(path)

        return Document(
            content=content,
            file_path=str(path),
            file_type=extension,
            metadata={}
        )

    elif extension in [".txt", ".md"]:

        content = load_text(path)

        return Document(
            content=content,
            file_path=str(path),
            file_type=extension,
            metadata={}
        )

    else:

        raise ValueError(
            f"Unsupported file type: {extension}"
        )

In [6]:
from pathlib import Path

test_file = Path(
    "/workspace/data/documents/test.txt"
)

test_file.write_text(
    """Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.
""",
    encoding="utf-8"
)

print(test_file)

/workspace/data/documents/test.txt


In [7]:
document = load_document(test_file)

print("File:", document.file_path)
print("Type:", document.file_type)
print()
print(document.content)

File: /workspace/data/documents/test.txt
Type: .txt

Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.



In [8]:
from dataclasses import dataclass


@dataclass
class Chunk:
    content: str
    chunk_id: str
    file_path: str
    file_type: str
    metadata: dict

In [9]:
def chunk_text(
    text,
    chunk_size=1000,
    chunk_overlap=200
):
    if not text.strip():
        return []

    chunks = []

    start = 0
    text_length = len(text)

    while start < text_length:

        end = min(
            start + chunk_size,
            text_length
        )

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start = end - chunk_overlap

    return chunks

In [10]:
text = """
Retrieval-Augmented Generation combines
retrieval systems with language models.

The retrieval component searches a knowledge
base for relevant information.

The language model then uses the retrieved
information to generate an answer.

Chunking is important because documents are
usually too large to send directly to an LLM.
"""

chunks = chunk_text(
    text,
    chunk_size=150,
    chunk_overlap=30
)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- CHUNK {i} ---")
    print(chunk)

Number of chunks: 3

--- CHUNK 0 ---
Retrieval-Augmented Generation combines
retrieval systems with language models.

The retrieval component searches a knowledge
base for relevant infor

--- CHUNK 1 ---
wledge
base for relevant information.

The language model then uses the retrieved
information to generate an answer.

Chunking is important because do

--- CHUNK 2 ---
unking is important because documents are
usually too large to send directly to an LLM.


In [11]:
def create_chunks(document):
    
    text_chunks = chunk_text(
        document.content,
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = []

    for index, content in enumerate(text_chunks):

        chunk = Chunk(
            content=content,
            chunk_id=f"{Path(document.file_path).name}-{index}",
            file_path=document.file_path,
            file_type=document.file_type,
            metadata={
                **document.metadata,
                "chunk_index": index
            }
        )

        chunks.append(chunk)

    return chunks

In [12]:
chunks = create_chunks(document)

print("Total chunks:", len(chunks))

for chunk in chunks:
    print("\nID:", chunk.chunk_id)
    print(chunk.content)

Total chunks: 1

ID: test.txt-0
Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.


In [13]:
def create_pdf_chunks(document):

    chunks = []

    for page in document.metadata["pages"]:

        page_number = page["page_number"]
        page_text = page["content"]

        page_chunks = chunk_text(
            page_text,
            chunk_size=1000,
            chunk_overlap=200
        )

        for index, content in enumerate(page_chunks):

            chunk = Chunk(
                content=content,

                chunk_id=(
                    f"{Path(document.file_path).name}"
                    f"-page-{page_number}"
                    f"-chunk-{index}"
                ),

                file_path=document.file_path,

                file_type=document.file_type,

                metadata={
                    "page_number": page_number,
                    "chunk_index": index
                }
            )

            chunks.append(chunk)

    return chunks

In [14]:
def create_chunks(document):

    if document.file_type == ".pdf":
        return create_pdf_chunks(document)

    text_chunks = chunk_text(
        document.content,
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = []

    for index, content in enumerate(text_chunks):

        chunks.append(
            Chunk(
                content=content,

                chunk_id=(
                    f"{Path(document.file_path).name}"
                    f"-chunk-{index}"
                ),

                file_path=document.file_path,

                file_type=document.file_type,

                metadata={
                    **document.metadata,
                    "chunk_index": index
                }
            )
        )

    return chunks

In [15]:
chunks = create_chunks(document)

print("Total chunks:", len(chunks))

for chunk in chunks:
    print(
        chunk.chunk_id,
        "→",
        len(chunk.content),
        "characters"
    )

Total chunks: 1
test.txt-chunk-0 → 326 characters


In [16]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4611.04it/s]


Embedding model loaded


In [17]:
test_embedding = embedding_model.encode(
    chunks[0].content
)

print("Embedding type:", type(test_embedding))
print("Embedding dimensions:", len(test_embedding))

Embedding type: <class 'numpy.ndarray'>
Embedding dimensions: 384


In [18]:
def embed_chunks(chunks, embedding_model):

    texts = [
        chunk.content
        for chunk in chunks
    ]

    embeddings = embedding_model.encode(
        texts,
        show_progress_bar=True
    )

    return embeddings

In [19]:
embeddings = embed_chunks(
    chunks,
    embedding_model
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.24it/s]

Number of embeddings: 1
Embedding dimensions: 384


In [20]:
import chromadb
from pathlib import Path


CHROMA_PATH = "/workspace/data/chroma_data"

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

print("ChromaDB initialized")

ChromaDB initialized


In [21]:
collection = chroma_client.get_or_create_collection(
    name="documents"
)

print("Collection:", collection.name)

Collection: documents
